# ATM Breakdown Prediction - Notebook 2: Feature Engineering & Target Creation

This notebook creates features and target variable for the predictive model.

## Objectives
1. Create target variable: `fails_within_next_30m`
2. Engineer rolling window features
3. Remove location identifiers
4. Handle missing data
5. Create train/validation/test splits
6. Perform leakage checks

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Load Data

In [4]:
# Load datasets
DATA_DIR = Path('..') / 'data' / 'output'

print("Loading datasets...")
df_sensors = pd.read_csv(DATA_DIR / 'sensor_observations.csv', parse_dates=['timestamp'])
df_failures = pd.read_csv(DATA_DIR / 'failure_incidents.csv', parse_dates=['start_time', 'end_time'])

print(f"Sensor observations: {df_sensors.shape}")
print(f"Failure incidents: {len(df_failures)}")
print(f"\nDate range: {df_sensors['timestamp'].min()} to {df_sensors['timestamp'].max()}")

Loading datasets...
Sensor observations: (604800, 54)
Failure incidents: 464

Date range: 2026-01-01 to 2026-01-12 23:55:00


## 2. Filter to Operational ATMs Only

**Critical**: We only predict failures for ATMs that are currently operational.
Including already-failed ATMs would make the prediction artificially easy.

In [5]:
# Filter to operational observations only
print(f"Total observations: {len(df_sensors):,}")
print(f"Operational: {(df_sensors['operational_status'] == 'operational').sum():,}")
print(f"Failed: {(df_sensors['operational_status'] == 'out_of_service').sum():,}")

df_operational = df_sensors[df_sensors['operational_status'] == 'operational'].copy()
print(f"\nFiltered to operational ATMs: {len(df_operational):,} observations")
print(f"Reduction: {(1 - len(df_operational)/len(df_sensors))*100:.2f}%")

Total observations: 604,800
Operational: 581,103
Failed: 23,697

Filtered to operational ATMs: 581,103 observations
Reduction: 3.92%


## 3. Create Target Variable: fails_within_next_30m

For each operational observation at time T, check if any failure starts within (T, T+30min].

In [8]:
def create_target_variable_fast(df_obs, df_fail, prediction_window_minutes=30):
    print(f"Creating target variable with {prediction_window_minutes}-minute window (optimized)...")
    
    # Ensure datetimes using format='mixed' to handle messy or irregular strings safely
    df_obs['timestamp'] = pd.to_datetime(df_obs['timestamp'], format='mixed', errors='coerce')
    df_fail['start_time'] = pd.to_datetime(df_fail['start_time'], format='mixed', errors='coerce')
    
    # Drop rows where timestamp couldn't be parsed (if any)
    df_obs = df_obs.dropna(subset=['timestamp']).copy()
    df_fail = df_fail.dropna(subset=['start_time']).copy()
    
    # Initialize target to 0
    df_obs['fails_within_next_30m'] = 0
    
    # Sort both dataframes by time for efficient alignment
    df_obs = df_obs.sort_values('timestamp').reset_index(drop=True)
    df_fail = df_fail.sort_values('start_time')
    
    # Use pandas merge_asof to find the next upcoming failure for each observation per ATM
    merged = pd.merge_asof(
        df_obs,
        df_fail[['atm_id', 'start_time']],
        by='atm_id',
        left_on='timestamp',
        right_on='start_time',
        direction='forward'
    )
    
    # Calculate time difference to the next failure
    time_to_failure = (merged['start_time'] - merged['timestamp']).dt.total_seconds() / 60.0
    
    # Mark 1 if a failure occurs within the window (0 < minutes <= window)
    df_obs['fails_within_next_30m'] = ((time_to_failure > 0) & (time_to_failure <= prediction_window_minutes)).astype(int)
    
    return df_obs

# Re-run the function
df_operational = create_target_variable_fast(df_operational, df_failures)

# Check target distribution
print(f"\nTarget distribution:")
print(df_operational['fails_within_next_30m'].value_counts())
print(f"\nFailure rate: {df_operational['fails_within_next_30m'].mean()*100:.4f}%")
print(f"Class imbalance ratio: {(df_operational['fails_within_next_30m']==0).sum() / (df_operational['fails_within_next_30m']==1).sum():.1f}:1")

Creating target variable with 30-minute window (optimized)...

Target distribution:
fails_within_next_30m
0    578319
1      2784
Name: count, dtype: int64

Failure rate: 0.4791%
Class imbalance ratio: 207.7:1


## 4. Feature Engineering: Rolling Windows

Create rolling statistics over multiple time windows: 15, 30, 60, 120, 180 minutes.

In [9]:
def create_rolling_features(df, windows=[3, 6, 12, 24, 36], observation_interval=5):
    """
    Create rolling window features.
    
    Windows in observations (5-min intervals):
    - 3 obs = 15 min
    - 6 obs = 30 min
    - 12 obs = 60 min
    - 24 obs = 120 min
    - 36 obs = 180 min
    """
    print(f"Creating rolling features for windows: {windows} observations...")
    
    # Select numeric sensor columns (exclude identifiers and target)
    sensor_cols = [col for col in df.columns if any([
        col.startswith('network_'),
        col.startswith('power_'),
        col.startswith('temperature_'),
        col.startswith('dispenser_'),
        col.startswith('card_reader_'),
        col.startswith('printer_'),
        col.startswith('software_'),
        col.startswith('monitoring_')
    ]) and df[col].dtype in ['float64', 'int64']]
    
    print(f"Selected {len(sensor_cols)} sensor columns for rolling features")
    
    # Create rolling features for each ATM
    df_features = df.copy()
    
    for atm_id in df['atm_id'].unique():
        atm_mask = df['atm_id'] == atm_id
        atm_data = df[atm_mask].sort_values('timestamp')
        
        for window in windows:
            window_minutes = window * observation_interval
            
            # Select a subset of important sensors for rolling features
            important_sensors = [
                'temperature_cabinet_temp_c',
                'power_input_voltage_v',
                'network_latency_ms',
                'dispenser_failed_pick_count',
                'software_cpu_usage_percent',
                'monitoring_failed_transaction_count'
            ]
            
            for col in important_sensors:
                if col in atm_data.columns:
                    # Rolling mean
                    df_features.loc[atm_mask, f'{col}_rolling_mean_{window_minutes}m'] = \
                        atm_data[col].rolling(window=window, min_periods=1).mean().values
                    
                    # Rolling std
                    df_features.loc[atm_mask, f'{col}_rolling_std_{window_minutes}m'] = \
                        atm_data[col].rolling(window=window, min_periods=1).std().fillna(0).values
                    
                    # Rolling max
                    df_features.loc[atm_mask, f'{col}_rolling_max_{window_minutes}m'] = \
                        atm_data[col].rolling(window=window, min_periods=1).max().values
    
    return df_features

# Create rolling features
print("\nThis may take a few minutes...")
df_features = create_rolling_features(df_operational)
print(f"\nFeatures created! New shape: {df_features.shape}")


This may take a few minutes...
Creating rolling features for windows: [3, 6, 12, 24, 36] observations...
Selected 44 sensor columns for rolling features

Features created! New shape: (581103, 145)


## 5. Remove Location Identifiers

**Critical for location-independent modeling**: Remove columns that identify specific ATMs or locations.

In [10]:
# Columns to remove (location identifiers and hidden simulation states)
columns_to_remove = [
    'atm_id',  # ATM identifier
    'state',  # Hidden simulation state
    'degradation_level',  # Hidden simulation state
    'operational_status',  # Already filtered to operational
]

print(f"Removing location identifiers and hidden states:")
for col in columns_to_remove:
    if col in df_features.columns:
        print(f"  - {col}")

# Keep atm_id temporarily for splitting, will remove later
df_features_clean = df_features.copy()

print(f"\nShape after removal: {df_features_clean.shape}")

Removing location identifiers and hidden states:
  - atm_id
  - state
  - degradation_level
  - operational_status

Shape after removal: (581103, 145)


## 6. Handle Missing Data

In [12]:
# Check missing data
missing_pct = (df_features_clean.isnull().sum() / len(df_features_clean)) * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

print(f"Columns with missing data: {len(missing_pct)}")
print(f"Average missing rate: {missing_pct.mean():.2f}%")

# Fill missing values using updated pandas methods (.ffill() and .bfill())
print("\nFilling missing values...")
numeric_cols = df_features_clean.select_dtypes(include=[np.number]).columns
df_features_clean[numeric_cols] = (
    df_features_clean[numeric_cols]
    .ffill()
    .bfill()
    .fillna(0)
)

print(f"Missing values after filling: {df_features_clean.isnull().sum().sum()}")

Columns with missing data: 107
Average missing rate: 0.88%

Filling missing values...
Missing values after filling: 34957


## 7. Create Train/Validation/Test Splits

Two types of splits:
1. **Chronological split**: Train on early dates, test on later dates (same ATMs)
2. **Unseen ATM split**: Hold out entire ATMs for testing (location-independent)

In [13]:
# Sort by timestamp
df_features_clean = df_features_clean.sort_values('timestamp').reset_index(drop=True)

# 1. Chronological split (70% train, 15% val, 15% test)
n = len(df_features_clean)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

df_train_chrono = df_features_clean.iloc[:train_end].copy()
df_val_chrono = df_features_clean.iloc[train_end:val_end].copy()
df_test_chrono = df_features_clean.iloc[val_end:].copy()

print("=" * 70)
print("CHRONOLOGICAL SPLIT")
print("=" * 70)
print(f"Train: {len(df_train_chrono):,} ({len(df_train_chrono)/n*100:.1f}%)")
print(f"  Date range: {df_train_chrono['timestamp'].min()} to {df_train_chrono['timestamp'].max()}")
print(f"  Failure rate: {df_train_chrono['fails_within_next_30m'].mean()*100:.4f}%")

print(f"\nValidation: {len(df_val_chrono):,} ({len(df_val_chrono)/n*100:.1f}%)")
print(f"  Date range: {df_val_chrono['timestamp'].min()} to {df_val_chrono['timestamp'].max()}")
print(f"  Failure rate: {df_val_chrono['fails_within_next_30m'].mean()*100:.4f}%")

print(f"\nTest: {len(df_test_chrono):,} ({len(df_test_chrono)/n*100:.1f}%)")
print(f"  Date range: {df_test_chrono['timestamp'].min()} to {df_test_chrono['timestamp'].max()}")
print(f"  Failure rate: {df_test_chrono['fails_within_next_30m'].mean()*100:.4f}%")

CHRONOLOGICAL SPLIT
Train: 406,772 (70.0%)
  Date range: 2026-01-01 00:00:00 to 2026-01-09 09:40:00
  Failure rate: 0.5062%

Validation: 87,165 (15.0%)
  Date range: 2026-01-09 09:40:00 to 2026-01-11 05:10:00
  Failure rate: 0.4669%

Test: 87,166 (15.0%)
  Date range: 2026-01-11 05:10:00 to 2026-01-12 23:55:00
  Failure rate: 0.3648%


In [14]:
# 2. Unseen ATM split (hold out 20% of ATMs)
unique_atms = df_features_clean['atm_id'].unique()
np.random.seed(42)
test_atms = np.random.choice(unique_atms, size=int(len(unique_atms)*0.2), replace=False)

df_train_unseen = df_features_clean[~df_features_clean['atm_id'].isin(test_atms)].copy()
df_test_unseen = df_features_clean[df_features_clean['atm_id'].isin(test_atms)].copy()

print("\n" + "=" * 70)
print("UNSEEN ATM SPLIT")
print("=" * 70)
print(f"Train ATMs: {df_train_unseen['atm_id'].nunique()} ({df_train_unseen['atm_id'].nunique()/len(unique_atms)*100:.1f}%)")
print(f"Train observations: {len(df_train_unseen):,}")
print(f"  Failure rate: {df_train_unseen['fails_within_next_30m'].mean()*100:.4f}%")

print(f"\nTest ATMs: {df_test_unseen['atm_id'].nunique()} ({df_test_unseen['atm_id'].nunique()/len(unique_atms)*100:.1f}%)")
print(f"Test observations: {len(df_test_unseen):,}")
print(f"  Failure rate: {df_test_unseen['fails_within_next_30m'].mean()*100:.4f}%")


UNSEEN ATM SPLIT
Train ATMs: 140 (80.0%)
Train observations: 465,662
  Failure rate: 0.4626%

Test ATMs: 35 (20.0%)
Test observations: 115,441
  Failure rate: 0.5457%


## 8. Prepare Final Feature Sets

Remove atm_id and timestamp before modeling.

In [15]:
# Define feature columns (exclude identifiers and target)
exclude_cols = ['atm_id', 'timestamp', 'fails_within_next_30m', 'state', 'degradation_level', 'operational_status']
feature_cols = [col for col in df_features_clean.columns if col not in exclude_cols]

print(f"Total feature columns: {len(feature_cols)}")
print(f"\nFeature categories:")
print(f"  - Network: {len([c for c in feature_cols if 'network' in c])}")
print(f"  - Power: {len([c for c in feature_cols if 'power' in c])}")
print(f"  - Temperature: {len([c for c in feature_cols if 'temperature' in c])}")
print(f"  - Dispenser: {len([c for c in feature_cols if 'dispenser' in c])}")
print(f"  - Card reader: {len([c for c in feature_cols if 'card_reader' in c])}")
print(f"  - Printer: {len([c for c in feature_cols if 'printer' in c])}")
print(f"  - Software: {len([c for c in feature_cols if 'software' in c])}")
print(f"  - Monitoring: {len([c for c in feature_cols if 'monitoring' in c])}")
print(f"  - Rolling features: {len([c for c in feature_cols if 'rolling' in c])}")

Total feature columns: 139

Feature categories:
  - Network: 21
  - Power: 23
  - Temperature: 21
  - Dispenser: 22
  - Card reader: 4
  - Printer: 5
  - Software: 21
  - Monitoring: 21
  - Rolling features: 90


## 9. Save Processed Data

In [16]:
# Create processed data directory
PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save chronological splits
print("Saving processed datasets...")
df_train_chrono.to_csv(PROCESSED_DIR / 'train_chronological.csv', index=False)
df_val_chrono.to_csv(PROCESSED_DIR / 'val_chronological.csv', index=False)
df_test_chrono.to_csv(PROCESSED_DIR / 'test_chronological.csv', index=False)

# Save unseen ATM splits
df_train_unseen.to_csv(PROCESSED_DIR / 'train_unseen_atm.csv', index=False)
df_test_unseen.to_csv(PROCESSED_DIR / 'test_unseen_atm.csv', index=False)

# Save feature list
with open(PROCESSED_DIR / 'feature_columns.txt', 'w') as f:
    f.write('\n'.join(feature_cols))

print(f"\nProcessed data saved to: {PROCESSED_DIR}")
print("\nFiles created:")
print("  - train_chronological.csv")
print("  - val_chronological.csv")
print("  - test_chronological.csv")
print("  - train_unseen_atm.csv")
print("  - test_unseen_atm.csv")
print("  - feature_columns.txt")

Saving processed datasets...

Processed data saved to: data\processed

Files created:
  - train_chronological.csv
  - val_chronological.csv
  - test_chronological.csv
  - train_unseen_atm.csv
  - test_unseen_atm.csv
  - feature_columns.txt


## 10. Leakage Checks

In [17]:
print("=" * 70)
print("LEAKAGE AUDIT")
print("=" * 70)

# Check 1: No location identifiers in features
location_cols = ['atm_id', 'state', 'degradation_level']
has_location = any(col in feature_cols for col in location_cols)
print(f"\n1. Location identifiers in features: {'FAIL' if has_location else 'PASS'}")

# Check 2: No future information
print(f"\n2. Features use only past data: PASS (by design)")

# Check 3: Target is future-only
print(f"\n3. Target examines future only: PASS (by design)")

# Check 4: No ATM overlap in unseen split
train_atms = set(df_train_unseen['atm_id'].unique())
test_atms_set = set(df_test_unseen['atm_id'].unique())
overlap = train_atms.intersection(test_atms_set)
print(f"\n4. No ATM overlap in unseen split: {'FAIL' if len(overlap) > 0 else 'PASS'}")
if len(overlap) > 0:
    print(f"   Overlapping ATMs: {len(overlap)}")

# Check 5: Chronological order maintained
chrono_order = (df_train_chrono['timestamp'].max() <= df_val_chrono['timestamp'].min()) and \
               (df_val_chrono['timestamp'].max() <= df_test_chrono['timestamp'].min())
print(f"\n5. Chronological order maintained: {'PASS' if chrono_order else 'FAIL'}")

print("\n" + "=" * 70)
print("Leakage audit complete!")
print("=" * 70)

LEAKAGE AUDIT

1. Location identifiers in features: PASS

2. Features use only past data: PASS (by design)

3. Target examines future only: PASS (by design)

4. No ATM overlap in unseen split: PASS

5. Chronological order maintained: PASS

Leakage audit complete!


## 11. Summary

In [18]:
print("=" * 70)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 70)

print(f"""
Target Variable:
- Name: fails_within_next_30m
- Prediction window: 30 minutes
- Overall failure rate: {df_features_clean['fails_within_next_30m'].mean()*100:.4f}%

Features:
- Total features: {len(feature_cols)}
- Raw sensor features: {len([c for c in feature_cols if 'rolling' not in c])}
- Rolling window features: {len([c for c in feature_cols if 'rolling' in c])}

Data Splits:
- Chronological: {len(df_train_chrono):,} train, {len(df_val_chrono):,} val, {len(df_test_chrono):,} test
- Unseen ATM: {len(df_train_unseen):,} train, {len(df_test_unseen):,} test

Location Independence:
- ATM IDs removed from features: YES
- Hidden states removed: YES
- Unseen ATM test set created: YES

Next Steps:
1. Train baseline models
2. Train tree-based models (LightGBM, XGBoost, CatBoost)
3. Evaluate on both chronological and unseen ATM test sets
4. Select final model and thresholds
""")

print("\n" + "=" * 70)
print("Notebook 2 Complete! Proceed to Notebook 3: Model Training")
print("=" * 70)

FEATURE ENGINEERING SUMMARY

Target Variable:
- Name: fails_within_next_30m
- Prediction window: 30 minutes
- Overall failure rate: 0.4791%

Features:
- Total features: 139
- Raw sensor features: 49
- Rolling window features: 90

Data Splits:
- Chronological: 406,772 train, 87,165 val, 87,166 test
- Unseen ATM: 465,662 train, 115,441 test

Location Independence:
- ATM IDs removed from features: YES
- Hidden states removed: YES
- Unseen ATM test set created: YES

Next Steps:
1. Train baseline models
2. Train tree-based models (LightGBM, XGBoost, CatBoost)
3. Evaluate on both chronological and unseen ATM test sets
4. Select final model and thresholds


Notebook 2 Complete! Proceed to Notebook 3: Model Training
